[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C33_Context_Memory_Course/01_token_budget/01_token_budget.ipynb)

# 01 · Token 预算与截断（token budget & truncation）

目标：用**纯标准库**从零写出上下文工程的地基——**token 计数 → 预算分配(含输出预留) → 头/尾/中间截断 → 优先级截断 + 消息钉住 → 完整组装走查**，全程用确定性逻辑 + `assert` 验证，**无需 API key**。

路线：近似计数器 → 预算分配器 → 位置截断 → 优先级截断器(保钉住/保近期/按优先级丢) → ✏️ 练习 → 📖 答案 → 🧪 真实 token 量级胶囊。

> 心智模型：**上下文窗口 = 固定预算；截断 = 超支时先砍谁**。先扣输出预留、先钉死必保项，再在剩余额度里按优先级从最不重要的砍起。

## 1 · 先学会数：近似 token 计数

一切从计数开始。真实系统用 `client.messages.count_tokens(model=..., messages=...)`（对目标模型精确，**绝不要用别家 tokenizer 如 tiktoken 估 Claude，会偏差 15–20%+**）。

本课用一个**确定性近似**：可重复、可断言，让我们专注「计数→预算→裁剪」的控制流。`count_messages` 还要给每条消息计入固定开销(角色标记等)。

In [ ]:
import json, re

def count_tokens(text):
    '''确定性近似: 按空白切词, 长词按字符折算。真实请用 messages.count_tokens。'''
    if not text:
        return 0
    return sum(max(1, (len(w) + 3) // 4) for w in text.split())

PER_MSG_OVERHEAD = 4   # 每条消息的固定开销(角色标记/分隔符), 逼近真实形状

def count_messages(messages):
    '''一组消息(role+content)的近似 token, 含每条固定开销。'''
    total = 0
    for m in messages:
        total += PER_MSG_OVERHEAD
        content = m['content'] if isinstance(m['content'], str) \
            else json.dumps(m['content'], ensure_ascii=False)
        total += count_tokens(content)
    return total

print('空串:', count_tokens(''))
print('hello world:', count_tokens('hello world'))
msgs = [{'role':'system','content':'you are a helpful assistant'},
        {'role':'user','content':'what is the capital of france'}]
print('两条消息(含开销):', count_messages(msgs))
# 不变量: 空串=0; 单调(更长不更少); 含每条开销
assert count_tokens('') == 0
assert count_tokens('a b c d e') >= count_tokens('a b c')
assert count_messages([{'role':'user','content':''}]) == PER_MSG_OVERHEAD  # 空内容也有开销
assert count_messages(msgs) == count_messages(msgs)                          # 确定性
print('✅ 计数器: 空串0、单调、每条含固定开销、确定性 —— 真实换成 count_tokens 即可')

## 2 · 再学会分：预算分配与输出预留

窗口是**输入和输出共享**的：`输入 token + max_tokens ≤ 窗口`。所以分配的**第一步永远是先扣掉输出预留**，再扣必保的系统/工具，剩下的才是留给历史/记忆/检索的额度。

下面的 `allocate` 把这套规则写死：返回各层额度与「留给历史」的可用值，并判断是否已经超支。

In [ ]:
def allocate(window, reserve_output, pinned_tokens, input_tokens):
    '''分层预算: 先扣输出预留, 再扣钉住项与本轮输入, 剩下的留给历史。
       返回 dict: 可放输入总额 / 必保(钉住+输入) / 留给历史 / 是否已超支。'''
    budget_for_input = window - reserve_output      # 第一步: 扣输出预留
    must_keep = pinned_tokens + input_tokens         # 钉住项 + 本轮输入, 必保
    for_history = budget_for_input - must_keep       # 剩下的才给历史
    return {
        'budget_for_input': budget_for_input,
        'must_keep': must_keep,
        'for_history': for_history,
        'overcommitted': for_history < 0,            # 连必保都装不下 = 设计就错了
    }

a = allocate(window=8000, reserve_output=1000, pinned_tokens=1500, input_tokens=300)
print('可放输入:', a['budget_for_input'], '| 必保:', a['must_keep'], '| 留给历史:', a['for_history'])
assert a['budget_for_input'] == 7000          # 8000 - 1000
assert a['must_keep'] == 1800                 # 1500 + 300
assert a['for_history'] == 5200               # 7000 - 1800
assert a['overcommitted'] is False
# 反例: 钉住项太大, 连必保都装不下 -> 应报超支
bad = allocate(window=2000, reserve_output=1000, pinned_tokens=1500, input_tokens=300)
assert bad['overcommitted'] is True and bad['for_history'] < 0
print('✅ 预算分配: 先扣输出预留, 再算留给历史的额度; 必保装不下能报超支')

## 3 · 超支了砍谁(一)：头/尾/中间截断

历史超出留给它的额度时就要截断。三种按位置的策略：**头部**(丢最旧, 常见默认)、**尾部**(丢最新, 一般不可取)、**中间**(保两端, 依据 Liu 2023《Lost in the Middle》: 模型对上下文两端利用率高、中段低)。

下面对一串「历史消息」按 token 预算做三种截断, 都保证结果 token ≤ 预算。

In [ ]:
def truncate_head(history, budget):
    '''丢最旧: 从尾部往前累加, 塞得下就保留 -> 保住最新的。'''
    kept, used = [], 0
    for m in reversed(history):
        t = count_messages([m])
        if used + t > budget:
            break
        kept.append(m); used += t
    return list(reversed(kept))

def truncate_tail(history, budget):
    '''丢最新: 从头部往后累加 -> 保住最旧的(保开头设定时偶用)。'''
    kept, used = [], 0
    for m in history:
        t = count_messages([m])
        if used + t > budget:
            break
        kept.append(m); used += t
    return kept

def truncate_middle(history, budget):
    '''保两端、裁中段: 交替从头、从尾取, 直到预算用尽 (对齐 U 形注意力)。'''
    if not history:
        return []
    i, j = 0, len(history) - 1
    take_head = True
    chosen = {}            # index -> msg
    used = 0
    while i <= j:
        idx = i if take_head else j
        t = count_messages([history[idx]])
        if used + t > budget:
            break
        chosen[idx] = history[idx]; used += t
        if take_head: i += 1
        else: j -= 1
        take_head = not take_head
    return [chosen[k] for k in sorted(chosen)]   # 复原原始顺序

hist = [{'role':'user','content':f'message number {i} with some filler words here'}
        for i in range(8)]
full = count_messages(hist)
budget = full // 2
h = truncate_head(hist, budget); t = truncate_tail(hist, budget); m = truncate_middle(hist, budget)
print('全量 token:', full, '| 预算:', budget)
print('头部截断保留:', [x['content'].split()[2] for x in h])   # 取 message 后的编号
print('尾部截断保留:', [x['content'].split()[2] for x in t])
print('中间截断保留:', [x['content'].split()[2] for x in m])
# 三者都必须 ≤ 预算
for name, r in [('head',h),('tail',t),('middle',m)]:
    assert count_messages(r) <= budget, name
# 头部截断保住最新一条; 尾部截断保住最旧一条
assert h[-1] == hist[-1] and t[0] == hist[0]
# 中间截断: 保住最旧和最新, 裁掉的是中段
assert m[0] == hist[0] and m[-1] == hist[-1] and len(m) < len(hist)
print('✅ 三种位置截断: 都 ≤ 预算; 头保新、尾保旧、中保两端 (Lost-in-the-Middle)')

## 4 · 超支了砍谁(二)：优先级截断 + 消息钉住

按位置截断假设「越旧越该丢」, 但第 3 轮的硬约束可能比第 30 轮的闲聊重要。更鲁棒: 给每条消息打**优先级**, **钉住项永不丢**, 其余从最低优先级开始丢、丢到塞下为止。

约定: 每条消息带 `priority`(越大越重要) 和 `pinned`(True=永不丢)。系统提示天然钉住。

In [ ]:
def truncate_by_priority(messages, budget):
    '''钉住项无条件保留; 其余按 (priority 高→低, 新→旧) 排序逐条塞入, 直到预算用尽。
       返回保留的消息(复原原始顺序)。保证: 钉住项全在 & 总 token ≤ budget。'''
    pinned = [m for m in messages if m.get('pinned')]
    used = count_messages(pinned)
    if used > budget:
        raise ValueError(f'仅钉住项就 {used} > 预算 {budget}: 设计错误, 钉住太多')
    # 其余候选: 带原始下标, 按优先级高→低、同级新→旧
    cand = [(i, m) for i, m in enumerate(messages) if not m.get('pinned')]
    cand.sort(key=lambda im: (im[1].get('priority', 0), im[0]), reverse=True)
    keep_idx = set(i for i, m in enumerate(messages) if m.get('pinned'))
    for i, m in cand:
        t = count_messages([m])
        if used + t <= budget:
            keep_idx.add(i); used += t
        # 不 break: 后面可能有更小、塞得下的消息(尽量填满)
    return [messages[i] for i in sorted(keep_idx)]

convo = [
    {'role':'system','content':'always answer in USD','pinned':True,'priority':100},
    {'role':'user','content':'hi there how are you doing today','priority':1},     # 低: 寒暄
    {'role':'user','content':'my hard budget limit is 5000 dollars','pinned':True,'priority':90},
    {'role':'assistant','content':'noted your limit is 5000','priority':5},
    {'role':'user','content':'what laptops fit my needs and budget','priority':50},
    {'role':'assistant','content':'here are three options under budget','priority':50},
]
full = count_messages(convo)
budget = full - 6           # 略紧: 必须丢掉至少一条低优先级
kept = truncate_by_priority(convo, budget)
kept_contents = [m['content'] for m in kept]
print('全量:', full, '预算:', budget, '保留条数:', len(kept), '/', len(convo))
# 不变量1: 所有钉住项都在
for m in convo:
    if m.get('pinned'):
        assert m['content'] in kept_contents, '钉住项被丢了!'
# 不变量2: 总 token ≤ 预算
assert count_messages(kept) <= budget
# 不变量3: 被丢的是最低优先级(寒暄), 而非按位置丢最旧
assert 'hi there how are you doing today' not in kept_contents
print('✅ 优先级截断: 钉住项全在、总token≤预算、先丢最低优先级(而非最旧)')

## 5 · 把它跑起来：完整的预算-截断走查

拼成一条**可复现的流水线**: 扣输出预留 → 扣钉住项与本轮输入 → 算留给历史的额度 → 历史超额则优先级截断 → 断言总输入 ≤ 可放预算。这就是后面所有模块的组装骨架。

In [ ]:
def assemble_context(system, tools, history, user_input, window, reserve_output):
    '''组装一次请求的上下文, 必要时截断历史。返回 (最终messages, 诊断dict)。'''
    # system 与 tools 钉住; user_input 必保
    sys_msg = {'role':'system','content':system,'pinned':True,'priority':100}
    tool_msg = {'role':'system','content':tools,'pinned':True,'priority':100}
    in_msg = {'role':'user','content':user_input,'priority':80}   # 当前输入, 高优先级
    pinned_tokens = count_messages([sys_msg, tool_msg])
    input_tokens = count_messages([in_msg])
    plan = allocate(window, reserve_output, pinned_tokens, input_tokens)
    if plan['overcommitted']:
        raise ValueError('必保部分已超预算: 缩短系统提示或加大窗口')
    # 历史在「留给历史」额度内做优先级截断
    kept_history = truncate_by_priority(history, plan['for_history'])
    final = [sys_msg, tool_msg] + kept_history + [in_msg]
    diag = {**plan, 'final_input_tokens': count_messages(final),
            'history_kept': len(kept_history), 'history_total': len(history)}
    return final, diag

history = [{'role':'user','content':f'turn {i} some conversational filler text here','priority':10+i}
           for i in range(12)]
final, diag = assemble_context(
    system='you are a shopping assistant',
    tools='[search_products, get_price]',
    history=history, user_input='recommend a laptop',
    window=200, reserve_output=40)   # 故意收紧, 逼历史做截断
print('留给历史:', diag['for_history'], '| 历史保留:', diag['history_kept'], '/', diag['history_total'])
print('最终输入 token:', diag['final_input_tokens'], '| 可放输入预算:', diag['budget_for_input'])
# 三条验收不变量
assert diag['final_input_tokens'] <= diag['budget_for_input']          # ① 塞得下
assert final[0]['content'] == 'you are a shopping assistant'           # ② 系统提示在(钉住)
assert final[-1]['content'] == 'recommend a laptop'                    # 当前输入在
assert diag['history_kept'] < diag['history_total']                    # 确实截断了
print('✅ 完整走查: 总输入≤预算、系统提示与当前输入都在、历史被正确截断')

---
## ✏️ 练习 1：带角色权重的 token 计数

真实里不同角色的「每条开销」可能不同(如工具结果块更重)。实现 `count_weighted(messages, overhead_by_role)`: 在 `count_tokens(content)` 基础上, 每条消息加 `overhead_by_role[role]` 的开销(缺省角色用 4)。

返回总 token。

In [ ]:
def count_weighted(messages, overhead_by_role):
    # TODO: 对每条消息: 取 content 的 count_tokens + 该 role 的开销(缺省 4),
    #       content 非字符串时先 json.dumps(..., ensure_ascii=False); 求和返回
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ow = {'system': 8, 'user': 4, 'tool': 10}
ms = [{'role':'system','content':'sys'},
      {'role':'user','content':'hello world'},
      {'role':'tool','content':'result'},
      {'role':'assistant','content':'ok'}]   # assistant 不在表里 -> 缺省 4
exp = (count_tokens('sys')+8) + (count_tokens('hello world')+4) \
      + (count_tokens('result')+10) + (count_tokens('ok')+4)
assert count_weighted(ms, ow) == exp
assert count_weighted([], ow) == 0
print('✅ 练习 1 通过: 按角色加权的 token 计数')

## ✏️ 练习 2：预算分配——把剩余额度切给历史与检索

扩展分配器: 留给历史的额度还要再切一块给检索。实现 `split_remaining(for_history, retrieval_ratio)`: 把 `for_history` 按比例切成 (历史额度, 检索额度), 检索额度 = `floor(for_history * retrieval_ratio)`, 其余给历史。

返回 `(hist_budget, retr_budget)`; 若 `for_history <= 0` 返回 `(0, 0)`。

In [ ]:
import math
def split_remaining(for_history, retrieval_ratio):
    # TODO: for_history<=0 -> (0,0); 否则 retr = floor(for_history*ratio),
    #       hist = for_history - retr; 返回 (hist, retr)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert split_remaining(1000, 0.3) == (700, 300)
assert split_remaining(1001, 0.3) == (701, 300)   # floor(300.3)=300
assert split_remaining(0, 0.3) == (0, 0)
assert split_remaining(-50, 0.3) == (0, 0)
h, r = split_remaining(1000, 0.3)
assert h + r == 1000                               # 不丢额度
print('✅ 练习 2 通过: 剩余额度按比例切给历史与检索, 不丢额度')

## ✏️ 练习 3：截断策略——保最近 N 条 + 钉住项

实现 `keep_recent_n(messages, n)`: 保留**所有钉住项** + **最近 n 条非钉住消息**, 复原原始顺序返回。

(这是滑动窗口截断 + 钉住的结合; n 大于非钉住数时全保留。)

In [ ]:
def keep_recent_n(messages, n):
    # TODO: 钉住项全保留; 非钉住消息里取最近 n 条(按原始顺序中靠后的);
    #       合并后按原始下标排序返回
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
msgs3 = [
    {'role':'system','content':'S','pinned':True},
    {'role':'user','content':'a'},
    {'role':'user','content':'b'},
    {'role':'user','content':'c','pinned':True},   # 钉住, 不占 n 名额
    {'role':'user','content':'d'},
    {'role':'user','content':'e'},
]
out = [m['content'] for m in keep_recent_n(msgs3, 2)]
# 钉住 S,c 必在; 非钉住最近2条是 d,e; 顺序复原
assert out == ['S','c','d','e'], out
# n 很大 -> 全保留
assert [m['content'] for m in keep_recent_n(msgs3, 99)] == ['S','a','b','c','d','e']
print('✅ 练习 3 通过: 保钉住项 + 最近 N 条非钉住, 顺序复原')

## ✏️ 练习 4：优先级丢弃——丢到塞得下，且不浪费空间

实现 `drop_until_fit(messages, budget)`: 返回 `(kept, dropped_contents)`。规则同 worked 4 的优先级截断(钉住项必留, 其余按 priority 高→低、同级新→旧 尽量塞入), 但还要返回**被丢弃消息的 content 列表**。

并验证一条更强的不变量: **没有被丢弃的消息能在不超预算的情况下重新塞回**(空间已尽量用满)。

In [ ]:
def drop_until_fit(messages, budget):
    # TODO: 复用 worked 4 的思路: 钉住项必留; 其余按(priority高→低, 新→旧)逐条尝试塞入;
    #       返回 (保留消息按原始顺序, 被丢弃消息的 content 列表)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
conv4 = [
    {'role':'system','content':'sys prompt','pinned':True,'priority':100},
    {'role':'user','content':'aaaa bbbb cccc dddd','priority':10},
    {'role':'user','content':'short','priority':20},
    {'role':'user','content':'eeee ffff gggg hhhh','priority':5},
]
B = count_messages([conv4[0]]) + count_messages([conv4[2]]) + 2  # 放得下 sys+short, 余 2
kept, dropped = drop_until_fit(conv4, B)
kc = [m['content'] for m in kept]
assert 'sys prompt' in kc and 'short' in kc          # 钉住 + 最高优先级保住
assert count_messages(kept) <= B
# 更强不变量: 每个被丢的, 单独塞回都会超预算(空间没浪费)
used = count_messages(kept)
for d in dropped:
    dmsg = next(m for m in conv4 if m['content'] == d)
    assert used + count_messages([dmsg]) > B, f'{d} 本可塞回, 空间被浪费了'
print('✅ 练习 4 通过: 丢到塞得下、钉住项保住、且空间尽量用满')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def count_weighted(messages, overhead_by_role):
    total = 0
    for m in messages:
        content = m['content'] if isinstance(m['content'], str) \
            else json.dumps(m['content'], ensure_ascii=False)
        total += count_tokens(content) + overhead_by_role.get(m['role'], 4)
    return total

In [ ]:
# 练习 2 参考答案
import math
def split_remaining(for_history, retrieval_ratio):
    if for_history <= 0:
        return (0, 0)
    retr = math.floor(for_history * retrieval_ratio)
    return (for_history - retr, retr)

In [ ]:
# 练习 3 参考答案
def keep_recent_n(messages, n):
    pinned_idx = [i for i, m in enumerate(messages) if m.get('pinned')]
    non_pinned_idx = [i for i, m in enumerate(messages) if not m.get('pinned')]
    recent = non_pinned_idx[-n:] if n > 0 else []
    keep = set(pinned_idx) | set(recent)
    return [messages[i] for i in sorted(keep)]

In [ ]:
# 练习 4 参考答案
def drop_until_fit(messages, budget):
    pinned = [m for m in messages if m.get('pinned')]
    used = count_messages(pinned)
    keep_idx = set(i for i, m in enumerate(messages) if m.get('pinned'))
    cand = [(i, m) for i, m in enumerate(messages) if not m.get('pinned')]
    cand.sort(key=lambda im: (im[1].get('priority', 0), im[0]), reverse=True)
    for i, m in cand:
        t = count_messages([m])
        if used + t <= budget:
            keep_idx.add(i); used += t
    kept = [messages[i] for i in sorted(keep_idx)]
    dropped = [m['content'] for i, m in enumerate(messages) if i not in keep_idx]
    return kept, dropped

---
## 🧪 真实数据胶囊：真实 token 量级与窗口算术

下面是**贴近真实**的数字: 真实模型窗口(如 200K)、典型系统提示/工具/单轮的 token 量级, 以及 Messages API 的 `usage` 形状。我们用本课的计数与预算逻辑去算「聊到第几轮该压缩」, 体会真实工程里的窗口算术。

> 形状对照: 真实里 `client.messages.count_tokens(...)` 返回 `.input_tokens`; `client.messages.create(...)` 的响应有 `usage.input_tokens / usage.output_tokens`, 且 `stop_reason=='max_tokens'` 表示输出被截断。

In [ ]:
# 贴近真实的量级(单位: token)。真实数字用 messages.count_tokens 量得
REAL = {
    'window': 200_000,        # 典型窗口
    'system_prompt': 1_200,   # 一份不算短的系统提示
    'tools': 2_500,           # 若干工具的 JSON schema
    'per_turn': 350,          # 平均每轮(user+assistant)
    'reserve_output': 4_000,  # 给输出预留
}

def turns_until_compaction(real, fill_ratio=0.8):
    '''算: 聊多少轮后, 输入达到窗口的 fill_ratio, 该触发压缩。'''
    budget_for_input = real['window'] - real['reserve_output']
    trigger_at = int(budget_for_input * fill_ratio)
    fixed = real['system_prompt'] + real['tools']
    room_for_history = trigger_at - fixed
    return room_for_history // real['per_turn']

n = turns_until_compaction(REAL)
print(f'窗口 {REAL["window"]:,}, 预留输出 {REAL["reserve_output"]:,}')
print(f'约第 {n} 轮对话后, 输入达窗口 80%, 该触发 compaction(模块02)')
assert n > 0 and isinstance(n, int)
# 模拟一次真实 usage 的成本核对: 输入+输出都占用、都计费
fake_usage = {'input_tokens': 50_000, 'output_tokens': 800}
assert fake_usage['input_tokens'] + fake_usage['output_tokens'] <= REAL['window']
print(f'单次 usage: 输入 {fake_usage["input_tokens"]:,} + 输出 {fake_usage["output_tokens"]} 占用窗口')
print('✅ 胶囊: 用本课逻辑算真实量级 -> 知道何时该压缩、输入输出共享窗口')

**🧪 胶囊练习**: 实现 `bytes_until_overflow(real, current_input_tokens)`: 给定当前已用输入 token, 返回「还能再聊多少轮才会撑爆可放输入预算」(`(可放输入 - current) // per_turn`, 不足一轮取 0)。

In [ ]:
def turns_left(real, current_input_tokens):
    # TODO: budget = window - reserve_output; left = budget - current_input_tokens;
    #       返回 max(0, left // per_turn)
    raise NotImplementedError

In [ ]:
# 自测
assert turns_left(REAL, 100_000) == (200_000-4_000-100_000) // 350
assert turns_left(REAL, 196_000) == 0      # 已经几乎满了
print('还能聊', turns_left(REAL, 100_000), '轮; 已满时返回', turns_left(REAL, 196_000))
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def turns_left(real, current_input_tokens):
    budget = real['window'] - real['reserve_output']
    left = budget - current_input_tokens
    return max(0, left // real['per_turn'])

---
## 🔧 旁注：对应的真实 Anthropic 调用 + 无 key 回退

本课验证过的预算-截断 scaffold, 换成真实 Anthropic 只是把「数」和「发」两处换掉(伪代码, **需 API key; 无 key 时下方适配自动回退 MockLLM**)：

```python
import anthropic
client = anthropic.Anthropic()                       # 读 ANTHROPIC_API_KEY
# 1) 真实计数(替换近似 count_tokens):
n = client.messages.count_tokens(
        model='claude-sonnet-4-6', system=SYSTEM, messages=history).input_tokens
# 2) 本课的截断器原样可用:
history = truncate_by_priority(history, budget=WINDOW - MAX_OUT - n_pinned)
# 3) 发请求(system 字段天然钉在最前、不参与历史裁剪):
resp = client.messages.create(model='claude-sonnet-4-6', max_tokens=MAX_OUT,
                              system=SYSTEM, messages=history)
if resp.stop_reason == 'max_tokens':
    ...   # 输出被截断: max_tokens 留少了, 或改用流式
```

对应关系: 近似 `count_tokens` ↔ `messages.count_tokens(...).input_tokens`、组装的 `system+messages` ↔ Messages API 同名字段、「输出预留」↔ `max_tokens`、「钉住系统提示」↔ `system` 字段天然在前。**截断/钉住/预算逻辑一字不改**——这就是「scaffold 可迁移」。

In [ ]:
# 可跑的「无 key 自动回退」适配: 有 key 用真实 Claude, 没有就回退 MockLLM (确定性)
import os
def get_llm():
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            client = anthropic.Anthropic()
            class RealLLM:
                def complete(self, messages):
                    sys_txt = ''.join(m['content'] for m in messages if m['role']=='system')
                    turns = [m for m in messages if m['role'] != 'system']
                    r = client.messages.create(model='claude-sonnet-4-6', max_tokens=1024,
                                               system=sys_txt or None, messages=turns)
                    return ''.join(b.text for b in r.content if b.type=='text')
            return RealLLM(), 'real(claude-sonnet-4-6)'
        except Exception as e:
            print('真实 API 不可用, 回退 Mock:', type(e).__name__)
    class MockLLM:
        def complete(self, messages):
            last = messages[-1]['content'] if messages else ''
            return '今天晴，26°C。' if '天气' in str(last) else '(mock 回答)'
    return MockLLM(), 'mock'

llm, kind = get_llm()
print('当前 LLM:', kind)   # 本环境无 key -> mock
# 用上面验证过的组装结果发一次(这里直接喂个最小 messages)
ans = llm.complete([{'role':'system','content':'you are helpful'},
                    {'role':'user','content':'今天天气？'}])
assert kind in ('mock','real(claude-sonnet-4-6)') and isinstance(ans, str) and ans
print('回答:', ans)
print('✅ 适配就位: 有 key 真 Claude、无 key 回退 Mock —— 整本 notebook 绝不阻断')

### 小结
- **先学会数**: token 决定塞不塞得下与花多少钱; 数 Claude 用 `messages.count_tokens`, **绝不用 tiktoken**。
- **先扣输出预留**: `输入 + max_tokens ≤ 窗口`; 分配第一步永远是减去 `max_tokens`。
- **位置截断**: 头(丢旧·常用)/尾(丢新·少用)/中(保两端·Lost-in-the-Middle); 但位置≠重要性。
- **优先级截断 + 钉住**: 钉住项永不丢, 其余按优先级从最低丢起、丢到塞下且不浪费空间。
- **验收不变量**: 截断后 ① 钉住项全在 ② 总输入 ≤ 窗口−输出预留 ③ 空间尽量用满。

下一站: **模块 02 · Compaction 与摘要** —— 对最大的占用源(不断增长的历史), 把「丢弃」换成「压缩成摘要」, 腾空间又不失忆。